# 12.d — Ad hoc test: does the effect of the AMOC weakening depend on the period of the ERA5 reference?

**Ad hoc test, outside the suitability chain**: it reads the outputs of 1.1.d, 1.3.a, 2.g, 3.g and 12.c (and of 6.b and
12.a for the checks), and writes only in `<out_root>/test_era5_reference/<route>/<set>/`.

**Question.** 12.c found the suitable land at the cold margin very sensitive to the period of ERA5: north of 50° N it is
30 % larger for maize and 44 % for soybean with 1998–2014 than with 1982–1997 (mostly the observed warming). The
scenarios of 6.b are ERA5 1982–2014 + the change of EC-Earth3 (delta of 1.3.a). Does the effect of the AMOC weakening
(hosing − no hosing), above all north of 50° N, depend on where the reference puts the margin?

**Method.** Three references of the delta: **full** = ERA5 1982–2014 (1.1.d, = 6.b), **H1** = 1982–1997, **H2** =
1998–2014 (BIO of 12.c, code of 1.1.d). For each reference and run: delta BIO = reference + (run − EC-Earth3
historical 1981–2014), the raw BIO of 1.3.a and the delta function of 1.3.a / 12.a (ratio for precipitation, additive
where the historical < 0.1 mm/day): the same change of the model on three baseline climates. Final rule of 4.g / 6.b,
unchanged (SVM of 3.g trained on ERA5 1982–2014, cross-fitting, novelty against its training cells, ECOCROP TMIN, soil
≥ 10, low and high input). Effects as 6.b / 12.a: today's crop area suitable (points of the MIRCA2000 area of the crop)
and suitable land per region (%, cells assessable in both runs), for the AMOC weakening (hosing − no hosing, 2025 and
2050) and for the warming (no hosing − reference).

**Reading.** The effect of the weakening does not depend on the reference if H1, full and H2 give the same sign and
differ by less than the noise of the runs (SE of 12.a, read back). A larger difference would mean that the size of the
effect at the margin depends on the baseline: then the effect should be quoted with this range.

Limits: the change of EC-Earth3 is the same for the three references (only the level of the baseline changes); H1 and
H2 have 16–17 seasons, so their own sampling noise is part of the difference.

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import time
import numpy as np, pandas as pd, xarray as xr, joblib
import matplotlib.pyplot as plt
from pathlib import Path
import warnings; warnings.filterwarnings('ignore'); os.environ["PYTHONWARNINGS"] = "ignore"   # also in the joblib workers

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu
CFG = cu.load_config()

# ---- parameters: the final set of 1.3.a, 4.g and 6.b
CHOSEN_VARIANT = "A_60d"
CHOSEN_RAD = "no_rad"
CELL_AREA_THR = 0.0005
CROPS = ["Maize_rf", "Potato_rf", "Soybean_rf", "Rice_rf", "Wheat_rf", "Wheat-winter_rf", "Wheat-spring_rf"]
ERA5_NAME, ERA5_PERIOD = "ERA5", ("1981", "2014")
HIST = "gcm_historical"
RUNS = ["gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]
PAIRS = [("gcm_2025_hosing", "gcm_2025_nohosing"), ("gcm_2050_hosing", "gcm_2050_nohosing")]   # hosing - original
WARMING = [("gcm_2025_nohosing", "reference"), ("gcm_2050_nohosing", "reference")]
REFS = ["full", "H1", "H2"]
LEVELS = ["low", "high"]
PR_MIN = 0.1                      # mm/day: below, the precipitation change is additive (1.3.a)
KERNEL_EPS = 0.01
N_JOBS = min(5, os.cpu_count() or 1)
OVERWRITE = False
assert CHOSEN_RAD == "no_rad", "no_rad only (no rsds in the runs)"

SET = f"{CHOSEN_VARIANT}_{CHOSEN_RAD}"
ROUTE = cu.route(CELL_AREA_THR)
TAG = f"{ERA5_NAME}_{ERA5_PERIOD[0]}_{ERA5_PERIOD[1]}_ecearth3"
ERA5_DIR = cu.out_dir(CFG, "1_biovariables", f"{ERA5_NAME}_{ROUTE}", CHOSEN_VARIANT)
RAW = cu.out_dir(CFG, "1_biovariables", f"GCM_raw_{ROUTE}", SET)        # 1.3.a: raw BIO of the runs and of the historical
MODELS = cu.out_dir(CFG, "3_svm_train", f"{ERA5_NAME}_{ROUTE}", SET)
AMOC_DIR = cu.out_dir(CFG, "6_climate_soil_amoc", ROUTE, SET)            # 6.b (check)
SOIL_DIR = cu.out_dir(CFG, "2_soil_suitability")
HALVES = cu.out_dir(CFG, "test_era5_robustness", ROUTE, SET) / "bio"    # 12.c: BIO of the halves of ERA5 (read only)
VAR12A = cu.out_dir(CFG, "test_internal_variability", ROUTE, SET)       # 12.a: noise of the runs (read only)
OUT = cu.out_dir(CFG, "test_era5_reference", ROUTE, SET)                # the only folder written
BIO_OUT = OUT / "bio"; BIO_OUT.mkdir(exist_ok=True)

GRID = cu.svm_grid(CFG)
LAND = cu.land_mask(CFG).values
LAND_MHA = np.nan_to_num(cu.land_area_ha(CFG)) * LAND / 1e6
DT = cu.tmin_warming(CFG)
LAT2, LON2 = np.meshgrid(GRID["lat"], GRID["lon"], indexing="ij")
LON180 = np.mod(LON2 + 180, 360) - 180
REGIONS = {"global": LAND,                                                   # as 6.b
           "Europe": LAND & (LAT2 >= 35) & (LAT2 <= 72) & (LON180 >= -25) & (LON180 <= 45),
           "north of 50 N": LAND & (LAT2 > 50),
           "23.5 - 50 N": LAND & (LAT2 > 23.5) & (LAT2 <= 50),
           "tropics": LAND & (np.abs(LAT2) <= 23.5),
           "south of 23.5 S": LAND & (LAT2 < -23.5)}
SOIL = {lv: xr.open_dataset(SOIL_DIR / f"soil_suitability_rf_{lv}_ecearth3.nc").load() for lv in LEVELS}
era5_file = lambda crop: ERA5_DIR / f"BIOvariables_{TAG}_{crop}.nc"
ref_file = lambda ref, crop: era5_file(crop) if ref == "full" else HALVES / f"BIOvariables_ERA5_{ref}_{crop}.nc"
delta_file = lambda ref, run, crop: BIO_OUT / f"BIOvariables_delta_{ref}_{run}_{crop}.nc"
with xr.open_dataset(era5_file(CROPS[0])) as d0:
    COLS = [v for v in d0.data_vars if v.startswith("BIO") and not v.endswith("e") and v not in ("BIO03", "BIO08")]
PR_COLS = [c for c in COLS if c in ("BIO05", "BIO10")]
missing = [str(ref_file(r, c)) for r in REFS for c in CROPS if not ref_file(r, c).exists()]
assert not missing, f"BIO of the ERA5 halves missing (run 12.c first): {missing[:3]}"
print(f"set {SET} | references {REFS} | runs {RUNS} | written only in {OUT}")

## 1. Delta BIO of the four runs on each reference (delta of 1.3.a)

In [ ]:
def delta(e5, h, raw, cols, pr_cols):
    # 1.3.a: ERA5 + (run - historical), ratio for precipitation (additive where historical < PR_MIN), >= 0
    out = {}
    for c in cols:
        r = raw[c]
        if c in pr_cols:
            v = np.where(h[c] >= PR_MIN, e5[c] * r / np.where(h[c] >= PR_MIN, h[c], 1.0), e5[c] + (r - h[c]))
        else:
            v = e5[c] + (r - h[c])
        out[c] = np.clip(v, 0, None) if c in pr_cols else v
    return out


rows = []
for crop in CROPS:
    with xr.open_dataset(RAW / f"BIOvariables_raw_{HIST}_{crop}.nc") as h:
        HB = {c: h[c].values for c in COLS}
    for ref in REFS:
        with xr.open_dataset(ref_file(ref, crop)) as e:
            E = {c: e[c].values for c in COLS}
            CAL = {v: e[v].values for v in ("sowing_doy", "phase2_start_doy", "maturity_doy")}
        for run in RUNS:
            if delta_file(ref, run, crop).exists() and not OVERWRITE:
                continue
            with xr.open_dataset(RAW / f"BIOvariables_raw_{run}_{crop}.nc") as r:
                raw = {c: r[c].values for c in COLS}
            dv = delta(E, HB, raw, COLS, PR_COLS)
            ds = xr.Dataset({c: (("lat", "lon"), dv[c].astype("float32")) for c in COLS},
                            coords={"lat": GRID["lat"], "lon": GRID["lon"]},
                            attrs={"description": f"delta BIO: ERA5 {ref} + ({run} - {HIST}), delta of 1.3.a, {SET}"})
            for v in CAL:
                ds[v] = (("lat", "lon"), CAL[v])
            ds.to_netcdf(delta_file(ref, run, crop))
    print(f"{crop}: delta BIO written", flush=True)
# check: the full reference reproduces the delta BIO of 1.3.a
DELTA13 = cu.out_dir(CFG, "1_biovariables", f"GCM_delta_{ROUTE}", SET)
for crop in CROPS:
    for run in RUNS:
        f13 = DELTA13 / f"BIOvariables_delta_{run}_{crop}.nc"
        if f13.exists():
            with xr.open_dataset(f13) as a, xr.open_dataset(delta_file("full", run, crop)) as b:
                rows.append({"crop": crop, "run": run, "max |full - 1.3.a|": max(float(np.nanmax(np.abs(a[c].values - b[c].values)[LAND])) for c in COLS)})
if rows:
    print("full reference against the delta BIO of 1.3.a (should be ~0):")
    display(pd.DataFrame(rows).groupby("crop").max(numeric_only=True))
else:
    print("no delta BIO of 1.3.a: check skipped")

## 2. Final suitability of every reference and run (rule of 4.g and 6.b, function of 12.a)

In [ ]:
def suitability(crop, files):
    # {key of files: {"suit": {level: bool (lat, lon)}, "novel": bool}}, as 4.g + 6.b (function of 12.a)
    bundle = joblib.load(MODELS / f"svm_model_{TAG}_{crop}.joblib")
    cols, n1, n0 = bundle["bio_cols"], bundle["n_class1"], bundle["n_class0"]
    assert set(cols) <= set(COLS), (cols, COLS)
    tr = pd.read_csv(ERA5_DIR / f"training_dataset_{TAG}_{crop}.csv")
    ii = np.searchsorted(GRID["lat"], tr.lat.values - 1e-6); jj = np.searchsorted(GRID["lon"], tr.lon.values - 1e-6)
    with xr.open_dataset(era5_file(crop)) as ds:
        REF = cu.bio_array(ds, cols, CFG)[ii, jj]
    REF = REF[np.isfinite(REF).all(axis=1)]
    kfun = cu.kernel_support(bundle["model"])
    Fd = cu.crossfit_models(MODELS / f"folds_{TAG}_{crop}.joblib", tr[cols].values.astype(float),
                            tr["label"].values.astype(int), bundle, n_jobs=N_JOBS)
    ssr = {lv: np.where(LAND, np.nan_to_num(SOIL[lv][crop].values), 0.0) for lv in LEVELS}
    which = cu.ECOCROP_TMIN[crop][1]
    out = {}
    for key, path in files.items():
        with xr.open_dataset(path) as ds:
            A = cu.bio_array(ds, cols, CFG)
            tg = np.asarray(cu.growing_temperature(ds, which), float)
        X = A.reshape(-1, len(cols))
        ok = np.isfinite(X).all(axis=1) & LAND.ravel()
        p, nov = np.full(X.shape[0], np.nan), np.zeros(X.shape[0], bool)
        if ok.any():                                              # (a part without any season: all NaN)
            Xs = bundle["scaler"].transform(X[ok])
            p[ok] = bundle["model"].predict_proba(Xs)[:, 1]
            nov[ok] = (cu.mess(X[ok], REF)[0] < 0) | (cu.nt2(X[ok], REF) > 1) | (kfun(Xs) < KERNEL_EPS)
        p, nov = p.reshape(LAND.shape), nov.reshape(LAND.shape)
        g = cu.crossfit_proba(Fd, A[ii, jj])                      # training cells: the fold model that did not see them
        fin = np.isfinite(g)
        p[ii[fin], jj[fin]] = g[fin]
        p = cu.balanced_proba(p, n1, n0)
        nov &= np.isfinite(p)
        proba = np.where(nov, 0.0, p)
        clim = cu.climate_suitable(proba, tg, crop, DT) & np.isfinite(proba)
        out[key] = {"suit": {lv: LAND & cu.suitable(clim, ssr[lv]) for lv in LEVELS}, "novel": nov}
    return out


S = {}
for crop in CROPS:
    t0 = time.time()
    S[crop] = suitability(crop, {**{(ref, "reference"): ref_file(ref, crop) for ref in REFS},
                                 **{(ref, run): delta_file(ref, run, crop) for ref in REFS for run in RUNS}})
    print(f"{crop}: {len(S[crop])} climates in {(time.time() - t0) / 60:.1f} min", flush=True)

rows = []                                       # check: full against 6.b
for crop in CROPS:
    f6 = AMOC_DIR / f"climate_soil_scenarios_{crop}.nc"
    if not f6.exists():
        continue
    with xr.open_dataset(f6) as ds:
        for run in [r for r in RUNS if r in ds.scenario.values]:
            for lv in LEVELS:
                a = ds[f"suitable_{lv}"].sel(scenario=run).values == 1
                rows.append({"crop": crop, "run": run, "level": lv,
                             "cells differing": int((a != S[crop][("full", run)]["suit"][lv])[LAND].sum())})
if rows:
    print("full reference against 6.b (should be 0):")
    display(pd.DataFrame(rows).groupby("level")[["cells differing"]].max())
else:
    print("no 6.b output: check skipped")

## 3. Effect of the AMOC weakening and of the warming on each reference

Metrics of 6.b / 12.a. `spread` = largest − smallest effect over the three references; `SE runs` = noise of the runs
of 12.a (SE of the full effect) where its output exists; `same sign` = the three references agree.

In [ ]:
mha = lambda m: float(np.nansum(LAND_MHA * m))
pct = lambda a, b: 100 * (a - b) / b if b > 0.5 else np.nan        # < 0.5 Mha: not reported (as 6.b)
rows = []
for crop in CROPS:
    crop_ha = np.nan_to_num(cu.crop_mask(CFG, crop)["area_ha"].values)
    share = lambda m: 100 * crop_ha[m].sum() / crop_ha.sum()
    R = S[crop]
    for lv in LEVELS:
        for ref in REFS:
            for kind, pairs in (("AMOC weakening", PAIRS), ("warming", WARMING)):
                for a_, b_ in pairs:
                    A, B = R[(ref, a_)], R[(ref, b_)]
                    both = LAND & ~(A["novel"] | B["novel"])
                    m = {"today's crop area (points)": share(A["suit"][lv]) - share(B["suit"][lv])}
                    for reg, rm in REGIONS.items():
                        m[f"{reg} land (%)"] = pct(mha(A["suit"][lv] & both & rm), mha(B["suit"][lv] & both & rm))
                    for k, v in m.items():
                        rows.append({"crop": crop, "level": lv, "effect": kind, "pair": f"{a_} - {b_}", "metric": k,
                                     "reference": ref, "value": v})
            for k in ["reference", *RUNS]:
                rows.append({"crop": crop, "level": lv, "effect": "level", "pair": k, "metric": "north of 50 N land (Mha)",
                             "reference": ref, "value": mha(R[(ref, k)]["suit"][lv] & REGIONS["north of 50 N"])})
EFF = pd.DataFrame(rows).pivot_table(index=["crop", "level", "effect", "pair", "metric"], columns="reference",
                                     values="value", sort=False)[REFS]
EFF["spread"] = EFF.max(axis=1) - EFF.min(axis=1)
EFF["same sign"] = (np.sign(EFF[REFS]).nunique(axis=1) == 1) & (EFF[REFS].abs().min(axis=1) > 0)
f12 = VAR12A / "effect_vs_noise.csv"
if f12.exists():
    N = pd.read_csv(f12).set_index(["crop", "level", "pair", "metric"])["SE (noise)"]
    EFF["SE runs (12.a)"] = [N.get((c, l, p, m), np.nan) for c, l, e, p, m in EFF.index]
    EFF["spread / SE runs"] = EFF["spread"] / EFF["SE runs (12.a)"]
EFF.to_csv(OUT / "effect_by_reference.csv")
with pd.option_context("display.max_rows", 500, "display.width", 250):
    show = EFF.reset_index()
    show = show[show.metric.isin(["today's crop area (points)", "global land (%)", "north of 50 N land (%)", "north of 50 N land (Mha)"])]
    for lv in LEVELS[::-1]:
        print(f"Effects on the three ERA5 references, {lv} input")
        display(show[show.level == lv].drop(columns="level").set_index(["crop", "effect", "pair", "metric"]).round(2))

## 4. Figure: effect of the AMOC weakening in 2050 on the three references, high input

In [ ]:
LV = "high" if "high" in LEVELS else LEVELS[0]
h, o = PAIRS[-1]
mets = ["today's crop area (points)", "global land (%)", "north of 50 N land (%)"]
fig, axes = plt.subplots(1, len(mets), figsize=(6.5 * len(mets), 0.5 * len(CROPS) + 2), squeeze=False)
cols = {"full": "#4d4d4d", "H1": "#2166ac", "H2": "#b2182b"}
for ax, met in zip(axes[0], mets):
    for k, crop in enumerate(CROPS):
        key = (crop, LV, "AMOC weakening", f"{h} - {o}", met)
        if key not in EFF.index:
            continue
        r = EFF.loc[key]
        for n, ref in enumerate(REFS):
            ax.barh(k + (n - 1) * 0.27, r[ref], height=0.25, color=cols[ref], label=ref if k == 0 else None)
        se = r.get("SE runs (12.a)", np.nan)
        if np.isfinite(se):
            ax.errorbar(r["full"], k - 0.27, xerr=2 * se, color="k", capsize=3, lw=1)
    ax.axvline(0, color="grey", lw=0.8); ax.set_yticks(range(len(CROPS)))
    ax.set_yticklabels([c.replace("_rf", "") for c in CROPS], fontsize=9); ax.invert_yaxis()
    ax.grid(axis="x", alpha=0.3); ax.set_title(met, fontsize=10)
axes[0, 0].legend(fontsize=8, title="ERA5 reference")
fig.suptitle(f"Effect of the AMOC weakening, 2050 ({LV} input), on the three ERA5 references of the delta "
             f"(full 1982-2014, H1 1982-1997, H2 1998-2014); black: ± 2 SE of the runs (12.a)", fontsize=11)
fig.tight_layout(); fig.savefig(OUT / f"effect_by_reference_{LV}.png", dpi=100, bbox_inches="tight"); plt.show()
print("written:", sorted(p.name for p in OUT.iterdir()))